In [ ]:
# Project Setup and Imports
import os
import sys
import importlib
from pathlib import Path

repo_root = Path.cwd()
print(f"Working directory: {repo_root}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

scraper_dir = repo_root / 'scraper'
if str(scraper_dir) not in sys.path:
    sys.path.insert(0, str(scraper_dir))

print('Python version:', sys.version)

# Try to import the main scraper modules
for mod_name in ['pipeline', 'enrichment', 'scraper']:
    try:
        mod = importlib.import_module(mod_name)
        print(f'{mod_name}: OK -> {getattr(mod, "__file__", "<built-in>")}')
    except Exception as exc:
        print(f'{mod_name}: ERROR -> {exc}')


In [ ]:
# Explore Project Modules
import inspect
import pipeline
import enrichment
import scraper

print('PipelineOrchestrator available:', hasattr(pipeline, 'PipelineOrchestrator'))
print('BatchCriteria available:', hasattr(pipeline, 'BatchCriteria'))
print('Enrichment helpers available:', [name for name in dir(enrichment) if name.startswith('apply') or name.startswith('validate') or name.startswith('clean')][:20])

print('\nPipelineOrchestrator signature:')
print(inspect.signature(pipeline.PipelineOrchestrator))
print('\nBatchCriteria signature:')
print(inspect.signature(pipeline.BatchCriteria))


In [ ]:
# Run Project Functionality Examples
import os
import sys
import traceback

try:
    from pipeline import PipelineOrchestrator, BatchCriteria
except Exception as exc:
    print('Import failed:', exc)
    raise

criteria = BatchCriteria(
    locations=['Douglasville, GA'],
    beds_min=3,
    beds_max=4,
    baths_min=1.0,
    rent_min=2000,
    rent_max=2500,
    rent_floor=2000,
    rent_cap=2500,
    allowed_types={'SINGLE_FAMILY', 'TOWNHOMES'},
    target=3,
    past_days=90,
    limit=50,
    min_score=35,
)

print('Criteria prepared:', criteria)

try:
    orchestrator = PipelineOrchestrator(verbose=True)
    result = orchestrator.run(criteria, dry_run=True)
    print('Dry run result:', result.summary())
except Exception as exc:
    print('Dry run failed with exception:')
    traceback.print_exc()


In [ ]:
# Analyze Project Outputs
import json

# If the dry run produced a result object, inspect its fields.
try:
    print('Result object type:', type(result).__name__)
    print('Result attributes:', [name for name in dir(result) if not name.startswith('_')][:60])
    if hasattr(result, 'published'):
        print('published:', result.published)
    if hasattr(result, 'errors'):
        print('errors:', result.errors)
    if hasattr(result, 'summary'):
        print('summary():', result.summary())
except Exception as exc:
    print('Output inspection failed:', exc)


In [ ]:
# Run Unit Tests and Validation
import os
import subprocess
import sys
from pathlib import Path

print('Checking scraper tests directory...')
for path in [Path('scraper/tests'), Path('tests')]:
    print(path, 'exists:', path.exists())

try:
    proc = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'scraper/tests'], capture_output=True, text=True, cwd=repo_root)
    print('pytest exit code:', proc.returncode)
    print(proc.stdout[:4000])
    print(proc.stderr[:4000])
except Exception as exc:
    print('pytest execution failed:', exc)
